# NYC Rent-Stabilized Housing Tax Bill Scraper

This notebook downloads NYC Department of Finance property-tax statements and extracts the number shown next to **Rent Stabilization–Chg** for each Borough-Block-Lot (BBL).

## What you need to change

Before running the scraper, edit the three values in **Section 2: User settings**:

1. `INPUT_CSV` — the CSV containing your BBL list.
2. `STATEMENT_DATE` — the tax-statement date in `YYYYMMDD` format.
3. `OUTPUT_CSV` — the filename for the results.

The input CSV must contain a column named `bbl`. BBLs should be stored as 10-digit text values, including leading zeros when applicable.

The notebook supports checkpointing and resumes automatically when the selected output file already exists.


## 1. Install dependencies

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "pdfplumber", "pypdf", "requests", "tqdm"])

## 2. User settings

Edit only the values in the **USER SETTINGS** block below for a typical run.

### Statement date (`STATEMENT_DATE`)

Enter the desired NYC Department of Finance statement date as an eight-digit string in `YYYYMMDD` format. For example:

```python
STATEMENT_DATE = "20211120"
```

The date must correspond to a statement available through the NYC DOF statement-search service. Different statement dates may return different rent-stabilized apartment counts or no statement at all.

### BBL list (`INPUT_CSV`)

Provide a CSV with a column named `bbl`. Example:

```text
bbl
1002470001
1001007502
1009720001
```

### Output filename (`OUTPUT_CSV`)

Choose a descriptive filename so separate statement dates do not overwrite one another. For example:

```python
OUTPUT_CSV = "rent_stabilized_counts_2021_q3.csv"
```


In [ ]:
import os
import re

# ============================================================================
# USER SETTINGS — EDIT THESE THREE VALUES FOR YOUR OWN RUN
# ============================================================================

# 1) CSV containing the BBLs to scrape.
#    Required column: bbl
#    Recommended format: one 10-digit BBL per row, stored as text.
INPUT_CSV = "bbls.csv"

# 2) NYC DOF statement date in YYYYMMDD format.
#    Example: "20211120"
#    Use a date associated with the quarter or billing period you want to test.
STATEMENT_DATE = "20211120"

# 3) Output CSV filename.
#    Include the year/quarter or statement date to avoid overwriting another run.
OUTPUT_CSV = "rent_stabilized_counts_2021_q3.csv"

# ============================================================================
# ADVANCED SETTINGS — MOST USERS CAN LEAVE THESE UNCHANGED
# ============================================================================

STATEMENT_TYPE = "SOA"
BASE_URL = (
    "https://a836-edms.nyc.gov/dctm-rest/repositories/dofedmspts/StatementSearch"
    "?bbl={bbl}&stmtDate={stmtDate}&stmtType={stmtType}"
)

REQUEST_TIMEOUT = (5, 30)   # (connection timeout, read timeout), in seconds
RETRY_COUNT = 2
RETRY_DELAY = 0.5
WORKERS = 10                # Reduce if the server is unstable or rate-limiting requests
SAVE_EVERY = 100            # Save a checkpoint after this many completed BBLs
MIN_PDF_BYTES = 10_000      # Helps identify incomplete PDF downloads

# Validate user settings before any requests are sent.
if not re.fullmatch(r"\d{8}", STATEMENT_DATE):
    raise ValueError(
        "STATEMENT_DATE must be an 8-digit string in YYYYMMDD format, "
        "for example '20211120'."
    )

if not INPUT_CSV.lower().endswith(".csv"):
    raise ValueError("INPUT_CSV must point to a .csv file.")

if not OUTPUT_CSV.lower().endswith(".csv"):
    raise ValueError("OUTPUT_CSV must end with .csv.")

print(f"Input file     : {os.path.abspath(INPUT_CSV)}")
print(f"Output file    : {os.path.abspath(OUTPUT_CSV)}")
print(f"Statement date : {STATEMENT_DATE}")
print(f"Statement type : {STATEMENT_TYPE}")
print(f"Workers        : {WORKERS}")


## 3. Load and validate the BBL list

The input file must contain a column named `bbl`. The code normalizes numeric-looking values, removes blanks, checks for 10-digit BBLs, and removes duplicates.


In [ ]:
import pandas as pd

if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(
        f"Input file not found: {os.path.abspath(INPUT_CSV)}. "
        "Place the CSV beside the notebook or provide its full path in INPUT_CSV."
    )

df_bbls = pd.read_csv(INPUT_CSV, dtype=str)
df_bbls.columns = df_bbls.columns.str.strip().str.lower()

if "bbl" not in df_bbls.columns:
    raise ValueError(
        "The input CSV must contain a column named 'bbl'. "
        f"Columns found: {list(df_bbls.columns)}"
    )

# Clean values that may have been exported from spreadsheet software as 1002470001.0.
df_bbls["bbl"] = (
    df_bbls["bbl"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

df_bbls = df_bbls.dropna(subset=["bbl"])
df_bbls = df_bbls[df_bbls["bbl"] != ""]

invalid_bbls = df_bbls.loc[~df_bbls["bbl"].str.fullmatch(r"\d{10}"), "bbl"].tolist()
if invalid_bbls:
    preview = ", ".join(invalid_bbls[:10])
    raise ValueError(
        "Every BBL must contain exactly 10 digits. "
        f"Invalid example(s): {preview}"
    )

all_bbls = df_bbls["bbl"].drop_duplicates().tolist()

print(f"Valid unique BBLs: {len(all_bbls):,}")


## 4. Parsing functions

In [ ]:
import re, io, time, logging
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pdfplumber
from pypdf import PdfReader

logging.basicConfig(
    level=logging.WARNING,   # INFO produces too much output with 10 workers
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── Persistent session — one per thread (thread-local) ───────────────────────
import threading
_local = threading.local()

def _get_session() -> requests.Session:
    """Return a per-thread session so connections are not shared across threads."""
    if not hasattr(_local, "session"):
        session = requests.Session()
        adapter = HTTPAdapter(
            max_retries=Retry(total=0, raise_on_status=False),
            pool_connections=2,
            pool_maxsize=2,
        )
        session.mount("https://", adapter)
        _local.session = session
    return _local.session

# ── Regex patterns ────────────────────────────────────────────────────────────
RS_LABEL      = re.compile(r"Rent\s+Stabilization[-–]\s*Chg", re.IGNORECASE)
PAT_INLINE    = re.compile(r"Rent\s+Stabilization[-–]\s*Chg\s+(\d+)", re.IGNORECASE)
PAT_NEXT_LINE = re.compile(r"Rent\s+Stabilization[-–]\s*Chg\s*\n\s*(\d+)", re.IGNORECASE)


def fetch_pdf_bytes(bbl: str, date_override: str = None) -> bytes | None:
    date = date_override if date_override else STATEMENT_DATE
    url = BASE_URL.format(bbl=bbl, stmtDate=date, stmtType=STATEMENT_TYPE)
    session = _get_session()
    for attempt in range(1, RETRY_COUNT + 1):
        try:
            with session.get(url, timeout=REQUEST_TIMEOUT, stream=True) as resp:
                if resp.status_code == 404:
                    return None
                if resp.status_code != 200:
                    log.warning("BBL %s: HTTP %s", bbl, resp.status_code)
                    return None
                chunks = [c for c in resp.iter_content(chunk_size=65536) if c]
                pdf_bytes = b"".join(chunks)
                if pdf_bytes[:4] != b"%PDF":
                    return None
                if len(pdf_bytes) < MIN_PDF_BYTES:
                    log.warning("BBL %s: truncated PDF (%d bytes), retry %d/%d",
                                bbl, len(pdf_bytes), attempt, RETRY_COUNT)
                    if attempt < RETRY_COUNT:
                        time.sleep(RETRY_DELAY)
                    continue
                return pdf_bytes
        except (requests.exceptions.ChunkedEncodingError,
                requests.exceptions.ConnectionError):
            log.warning("BBL %s: incomplete read, retry %d/%d", bbl, attempt, RETRY_COUNT)
            if attempt < RETRY_COUNT:
                time.sleep(RETRY_DELAY)
        except requests.exceptions.Timeout:
            log.warning("BBL %s: timeout attempt %d/%d", bbl, attempt, RETRY_COUNT)
            if attempt < RETRY_COUNT:
                time.sleep(RETRY_DELAY)
        except requests.RequestException as exc:
            log.warning("BBL %s: %s", bbl, exc)
            return None
    return None


def _get_full_text(pdf_bytes: bytes, extractor: str = "plumber") -> str:
    text = ""
    if extractor == "plumber":
        try:
            with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
                for page in pdf.pages:
                    t = page.extract_text(x_tolerance=3, y_tolerance=3)
                    if t:
                        text += t + "\n"
        except Exception:
            pass
    else:
        try:
            reader = PdfReader(io.BytesIO(pdf_bytes))
            for page in reader.pages:
                t = page.extract_text()
                if t:
                    text += t + "\n"
        except Exception:
            pass
    return text


def _inline_count(text: str) -> int | None:
    hits = [int(n) for n in PAT_INLINE.findall(text) if int(n) <= 9999]
    return sum(hits) if hits else None


def _nextline_count(text: str) -> int | None:
    hits = [int(n) for n in PAT_NEXT_LINE.findall(text) if int(n) <= 9999]
    return sum(hits) if hits else None


def _table_count(pdf_bytes: bytes) -> int | None:
    total, found = 0, False
    try:
        with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
            for page in pdf.pages:
                for table in page.extract_tables():
                    for row in table:
                        if not row:
                            continue
                        cells = [str(c).strip() if c else "" for c in row]
                        if not RS_LABEL.search(cells[0]):
                            continue
                        if len(cells) > 1 and re.fullmatch(r"\d+", cells[1]):
                            n = int(cells[1])
                            if 1 <= n <= 9999:
                                total += n
                                found = True
                                continue
                        candidates = [int(c) for c in cells
                                      if re.fullmatch(r"\d+", c) and 1 <= int(c) <= 9999]
                        if candidates:
                            total += min(candidates)
                            found = True
    except Exception:
        pass
    return total if found else None


def _window_count(text: str) -> int | None:
    lines = text.splitlines()
    total, found = 0, False
    for idx, line in enumerate(lines):
        if not RS_LABEL.search(line):
            continue
        window = " ".join(lines[max(0, idx-1): idx+3])
        candidates = [int(m) for m in re.findall(r"\b(\d{1,4})\b", window)
                      if 1 <= int(m) <= 9999]
        if candidates:
            total += min(candidates)
            found = True
    return total if found else None


def extract_rent_stabilized_count(pdf_bytes: bytes, bbl: str = "") -> int | None:
    plumber_text = _get_full_text(pdf_bytes, "plumber")
    pypdf_text   = _get_full_text(pdf_bytes, "pypdf")

    for label, text in [("plumber", plumber_text), ("pypdf", pypdf_text)]:
        result = _inline_count(text)
        if result is not None:
            return result

    for label, text in [("plumber", plumber_text), ("pypdf", pypdf_text)]:
        result = _nextline_count(text)
        if result is not None:
            return result

    result = _table_count(pdf_bytes)
    if result is not None:
        return result

    for label, text in [("plumber", plumber_text), ("pypdf", pypdf_text)]:
        result = _window_count(text)
        if result is not None:
            return result

    if RS_LABEL.search(plumber_text + pypdf_text):
        log.error("BBL %s: RS label found but count not extracted", bbl)
    return None


def process_bbl(bbl: str) -> dict:
    pdf_bytes = fetch_pdf_bytes(bbl)
    if pdf_bytes is None:
        return {"bbl": bbl, "rent_stabilized_apartments": None, "status": "no_pdf"}
    count = extract_rent_stabilized_count(pdf_bytes, bbl)
    if count is None:
        return {"bbl": bbl, "rent_stabilized_apartments": 0, "status": "no_rs_lines"}
    return {"bbl": bbl, "rent_stabilized_apartments": count, "status": "ok"}


print("Functions ready.")


## 5. Optional sanity check

These examples were used while developing the parser. They may fail when a different `STATEMENT_DATE` is selected because the statement may not exist for that date or the reported count may have changed.

Skip this section when using a different billing period, or replace the examples with BBLs and expected counts that you have manually verified for your selected statement date.


In [ ]:
STUY_EXPECTED = (
    105+94+85+89+95+88+88+105+105+105+96+103+103+95+89+88+105+107+107+107+210+
    105+89+91+89+97+95+105+105+105+96+96+95+96+93+98+91+98+91+103+103+103+103+
    103+105+103+103+96+95+105+105+96+102+105+95+105+103+95+91+95+95+95+105+95+
    97+95+95+95+97+91+95+95+105+103+105+89+89+97+105+105+107+89+105+104+94+93+
    105+105
)

sanity_cases = [
    ("1002470001", 490),
    ("1001007502", 900),
    ("1009720001", STUY_EXPECTED),
]

all_passed = True
for bbl, expected in sanity_cases:
    pdf_bytes = fetch_pdf_bytes(bbl)
    got = extract_rent_stabilized_count(pdf_bytes, bbl) if pdf_bytes else None
    ok = got == expected
    if not ok:
        all_passed = False
    print(f"[{'PASS' if ok else 'FAIL'}] BBL {bbl}: expected={expected:,}, got={got}")

print()
print("All sanity checks passed -- safe to run full scrape." if all_passed
      else "STOP: checks failed. Run cell 6 to debug.")

## 6. Debug – inspect raw text for any BBL

In [ ]:
DEBUG_BBL = "1009720001"  # change to any BBL you want to inspect

pdf_bytes = fetch_pdf_bytes(DEBUG_BBL)
if pdf_bytes:
    print(f"{len(pdf_bytes):,} bytes downloaded\n")
    text = _get_full_text(pdf_bytes, "plumber")
    rs_lines = [l for l in text.splitlines() if "stabiliz" in l.lower()]
    print(f"RS lines found (pdfplumber): {len(rs_lines)}")
    for l in rs_lines:
        print(" ", repr(l))
else:
    print("No PDF returned for", DEBUG_BBL)

## 7. Run the full scrape

The notebook saves progress to `OUTPUT_CSV`. If that same file already exists, BBLs already present in it are skipped and the run resumes.

Use a different output filename for each statement date or quarter. Reusing an existing filename from another date can incorrectly cause BBLs to be skipped.


In [ ]:
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

if os.path.exists(OUTPUT_CSV):
    df_done = pd.read_csv(OUTPUT_CSV, dtype=str)
    done_bbls = set(df_done["bbl"].tolist())
    results = df_done.to_dict("records")
    bbls_to_run = [b for b in all_bbls if b not in done_bbls]
    print(f"Resuming: {len(done_bbls):,} already done, {len(bbls_to_run):,} remaining")
else:
    results = []
    bbls_to_run = all_bbls
    print(f"Starting fresh: {len(bbls_to_run):,} BBLs to process")

errors = []

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(process_bbl, bbl): bbl for bbl in bbls_to_run}
    for i, future in enumerate(tqdm(as_completed(futures),
                                    total=len(bbls_to_run),
                                    desc="Scraping", unit="bbl")):
        row = future.result()
        results.append(row)
        if row["status"] != "ok":
            errors.append(row)
        if (i + 1) % SAVE_EVERY == 0:
            pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
            log.warning("Checkpoint saved (%d/%d)", len(results), len(all_bbls))

pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
print(f"Done. {len(results):,} total, {len(errors):,} issues.")

## 8. Summary

In [ ]:
df_results = pd.read_csv(OUTPUT_CSV, dtype=str)
df_results["rent_stabilized_apartments"] = pd.to_numeric(
    df_results["rent_stabilized_apartments"], errors="coerce")

print(f"Saved: {os.path.abspath(OUTPUT_CSV)}")
print(f"Total BBLs         : {len(df_results):,}")
print(f"RS apts > 0        : {(df_results['rent_stabilized_apartments'] > 0).sum():,}")
print(f"RS apts == 0       : {(df_results['rent_stabilized_apartments'] == 0).sum():,}")
print(f"No statement (NaN) : {df_results['rent_stabilized_apartments'].isna().sum():,}")
if 'status' in df_results.columns:
    print("\nStatus breakdown:")
    print(df_results["status"].value_counts().to_string())
display(df_results.head(10))